# STEP 31B — Ground-Truth Integrator (Auto / No-Edit Version)

This version is designed so that you **do not need to edit any path manually**.

When you press **Run All**, the notebook will:

1. Detect the BrainFMOps project folder automatically.
2. Find `evaluation_summary.csv` automatically.
3. Search common locations for an OASIS clinical spreadsheet.
4. If no clinical file is found, open a Windows file-selection window automatically.
5. Merge the clinical labels.
6. Export `evaluation_summary_with_labels.csv`.
7. Create audit and validation reports.

## Required evidence

A real OASIS clinical spreadsheet is still required. The notebook will never generate labels from model predictions.


In [ ]:
from pathlib import Path
import json
import os
import re
import math
import numpy as np
import pandas as pd

print("Environment ready.")


## 1. Automatically locate the BrainFMOps project


In [ ]:
def existing_paths(paths):
    return [p for p in paths if p.exists()]

home = Path.home()

project_candidates = existing_paths([
    Path.cwd().resolve(),
    Path.cwd().resolve(),
    home / "Desktop" / "BrainFMOps_Full_Evaluation",
    home / "Documents" / "BrainFMOps_Full_Evaluation",
    home / "Downloads" / "BrainFMOps_Full_Evaluation",
])

if project_candidates:
    PROJECT_ROOT = project_candidates[0]
else:
    # Fallback: current notebook directory
    PROJECT_ROOT = Path.cwd()

PREDICTION_FILE = PROJECT_ROOT / "evaluation_summary.csv"
OUTPUT_DIR = PROJECT_ROOT / "31B_GroundTruth_Integration"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "evaluation_summary_with_labels.csv"

print("Detected project root:", PROJECT_ROOT)
print("Prediction file:", PREDICTION_FILE)
print("Output directory:", OUTPUT_DIR)

if not PREDICTION_FILE.is_file():
    raise FileNotFoundError(
        f"evaluation_summary.csv was not found at {PREDICTION_FILE}"
    )


## 2. Automatically locate the OASIS clinical spreadsheet


In [ ]:
search_roots = [
    PROJECT_ROOT,
    Path.home() / "Downloads",
    Path.home() / "Desktop",
    Path.home() / "Documents",
]

patterns = [
    "*oasis*.csv", "*oasis*.xlsx", "*oasis*.xls",
    "*clinical*.csv", "*clinical*.xlsx", "*clinical*.xls",
    "*demographic*.csv", "*demographic*.xlsx", "*demographic*.xls",
    "*cdr*.csv", "*cdr*.xlsx", "*cdr*.xls",
]

excluded_names = {
    "evaluation_summary.csv",
    "evaluation_summary_with_labels.csv",
    "ground_truth_merge_audit.csv",
    "ground_truth_unmatched_predictions.csv",
    "ground_truth_unmatched_clinical.csv",
}

candidates = []
for root in search_roots:
    if not root.exists():
        continue
    for pattern in patterns:
        try:
            candidates.extend(root.rglob(pattern))
        except PermissionError:
            pass

candidates = sorted(set(
    p.resolve() for p in candidates
    if p.is_file() and p.name.lower() not in excluded_names
))

print("Clinical spreadsheet candidates found:")
for i, p in enumerate(candidates[:30]):
    print(f"  [{i}] {p}")

clinical_path = candidates[0] if candidates else None

if clinical_path is None:
    print("\nNo clinical spreadsheet was found automatically.")
    print("A Windows file-selection window will open now.")

    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)

        selected = filedialog.askopenfilename(
            title="Select the OASIS clinical spreadsheet",
            initialdir=str(Path.home() / "Downloads"),
            filetypes=[
                ("Clinical spreadsheets", "*.csv *.xlsx *.xls"),
                ("CSV files", "*.csv"),
                ("Excel files", "*.xlsx *.xls"),
                ("All files", "*.*"),
            ],
        )
        root.destroy()

        if selected:
            clinical_path = Path(selected)
    except Exception as exc:
        print("Automatic file-selection window could not be opened:", exc)

if clinical_path is None or not clinical_path.is_file():
    raise FileNotFoundError(
        "No OASIS clinical spreadsheet was selected. "
        "Place the file in Downloads or in the project folder, then run all cells again."
    )

print("\nSelected clinical file:", clinical_path)


## 3. Load files


In [ ]:
def load_table(path):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        for encoding in ["utf-8-sig", "utf-8", "latin-1"]:
            try:
                return pd.read_csv(path, encoding=encoding)
            except UnicodeDecodeError:
                continue
        return pd.read_csv(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported file type: {suffix}")

prediction_df = load_table(PREDICTION_FILE)
clinical_df = load_table(clinical_path)

print("Prediction shape:", prediction_df.shape)
print("Prediction columns:", list(prediction_df.columns))
print("Clinical shape:", clinical_df.shape)
print("Clinical columns:", list(clinical_df.columns))


## 4. Detect identifier and clinical-label columns


In [ ]:
def first_existing(columns, candidates):
    lookup = {str(c).strip().lower(): c for c in columns}
    for name in candidates:
        if name.lower() in lookup:
            return lookup[name.lower()]
    return None

prediction_id_col = first_existing(
    prediction_df.columns,
    ["case_id", "subject_id", "id", "subject", "mri_id"]
)

clinical_id_col = first_existing(
    clinical_df.columns,
    ["ID", "Subject ID", "Subject_ID", "subject_id", "subject", "case_id", "MRI ID"]
)

cdr_col = first_existing(
    clinical_df.columns,
    ["CDR", "cdr", "Clinical Dementia Rating"]
)

diagnosis_col = first_existing(
    clinical_df.columns,
    ["Diagnosis", "diagnosis", "Group", "group", "Class", "class",
     "Label", "label", "Clinical Group", "Dementia"]
)

if prediction_id_col is None:
    raise ValueError("Prediction identifier column was not found.")
if clinical_id_col is None:
    raise ValueError(
        "Clinical identifier column was not found. "
        f"Available columns: {list(clinical_df.columns)}"
    )
if cdr_col is None and diagnosis_col is None:
    raise ValueError(
        "Neither a CDR column nor an explicit diagnosis/group column was found. "
        f"Available columns: {list(clinical_df.columns)}"
    )

print("Prediction ID column:", prediction_id_col)
print("Clinical ID column:", clinical_id_col)
print("CDR column:", cdr_col)
print("Diagnosis/group column:", diagnosis_col)


## 5. Normalize OASIS subject identifiers


In [ ]:
def normalize_oasis_subject_id(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().upper()

    # OASIS-1: OAS1_0001_MR1 -> OAS1_0001
    match = re.search(r"(OAS1[_-]?\d{4})", text)
    if match:
        token = match.group(1).replace("-", "_")
        if "_" not in token:
            token = token[:4] + "_" + token[4:]
        return token

    # OASIS-2/3 style fallback
    match = re.search(r"(OAS\d+[_-]?\d+)", text)
    if match:
        return match.group(1).replace("-", "_")

    return text

prediction_df["subject_key"] = prediction_df[prediction_id_col].map(
    normalize_oasis_subject_id
)
clinical_df["subject_key"] = clinical_df[clinical_id_col].map(
    normalize_oasis_subject_id
)

display(prediction_df[[prediction_id_col, "subject_key"]].head())
display(clinical_df[[clinical_id_col, "subject_key"]].head())


## 6. Derive binary ground-truth labels


In [ ]:
def normalize_explicit_diagnosis(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().upper()

    cn_terms = {
        "CN", "NORMAL", "COGNITIVELY NORMAL", "CONTROL",
        "NONDEMENTED", "NON-DEMENTED", "HEALTHY", "0"
    }
    ad_terms = {
        "AD", "ALZHEIMER", "ALZHEIMER'S DISEASE",
        "DEMENTED", "DEMENTIA", "MILD DEMENTIA",
        "MODERATE DEMENTIA", "SEVERE DEMENTIA", "1"
    }

    if text in cn_terms:
        return "CN"
    if text in ad_terms:
        return "AD"
    return np.nan

if diagnosis_col is not None:
    clinical_df["ground_truth_explicit"] = clinical_df[diagnosis_col].map(
        normalize_explicit_diagnosis
    )
else:
    clinical_df["ground_truth_explicit"] = np.nan

if cdr_col is not None:
    clinical_df["cdr_numeric"] = pd.to_numeric(
        clinical_df[cdr_col], errors="coerce"
    )

    # NumPy 2.x-safe object Series: avoids mixing float NaN and strings in np.where.
    clinical_df["ground_truth_cdr"] = pd.Series(
        pd.NA, index=clinical_df.index, dtype="object"
    )
    valid_cdr = clinical_df["cdr_numeric"].notna()
    clinical_df.loc[
        valid_cdr & (clinical_df["cdr_numeric"] == 0),
        "ground_truth_cdr"
    ] = "CN"
    clinical_df.loc[
        valid_cdr & (clinical_df["cdr_numeric"] > 0),
        "ground_truth_cdr"
    ] = "AD"
else:
    clinical_df["cdr_numeric"] = np.nan
    clinical_df["ground_truth_cdr"] = pd.Series(
        pd.NA, index=clinical_df.index, dtype="object"
    )

clinical_df["ground_truth_derived"] = clinical_df["ground_truth_explicit"]
missing_explicit = clinical_df["ground_truth_derived"].isna()
clinical_df.loc[missing_explicit, "ground_truth_derived"] = clinical_df.loc[
    missing_explicit, "ground_truth_cdr"
]

print("Derived label counts:")
print(clinical_df["ground_truth_derived"].value_counts(dropna=False))


## 7. Resolve duplicate clinical records


In [ ]:
def resolve_subject_label(group):
    labels = sorted(set(group["ground_truth_derived"].dropna().astype(str)))
    cdr_values = sorted(set(group["cdr_numeric"].dropna().astype(float)))

    if len(labels) == 1:
        status = "resolved"
        label = labels[0]
    elif len(labels) == 0:
        status = "missing_label"
        label = np.nan
    else:
        status = "conflict"
        label = np.nan

    return pd.Series({
        "ground_truth": label,
        "clinical_resolution_status": status,
        "clinical_rows": len(group),
        "clinical_labels_observed": "|".join(labels),
        "cdr_values_observed": "|".join(map(str, cdr_values)),
    })

clinical_subjects = (
    clinical_df.groupby("subject_key", dropna=False)
    .apply(resolve_subject_label)
    .reset_index()
)

print(clinical_subjects["clinical_resolution_status"].value_counts())
display(clinical_subjects.head())


## 8. Merge labels into evaluation_summary.csv


In [ ]:
clinical_merge = clinical_subjects[
    [
        "subject_key", "ground_truth", "clinical_resolution_status",
        "clinical_rows", "clinical_labels_observed", "cdr_values_observed"
    ]
].rename(columns={"ground_truth": "ground_truth_clinical"})

merged = prediction_df.merge(
    clinical_merge,
    on="subject_key",
    how="left"
)

if "ground_truth" in merged.columns:
    merged["ground_truth_original"] = merged["ground_truth"]
    merged["ground_truth"] = merged["ground_truth_clinical"]
    merged = merged.drop(columns=["ground_truth_clinical"])
else:
    merged = merged.rename(columns={"ground_truth_clinical": "ground_truth"})

if "prediction" in merged.columns:
    merged["correct"] = np.where(
        merged["ground_truth"].notna(),
        merged["prediction"].astype(str).str.upper()
        == merged["ground_truth"].astype(str).str.upper(),
        np.nan
    )

print("Merged records:", len(merged))
print("Matched labels:", int(merged["ground_truth"].notna().sum()))
print("Unmatched labels:", int(merged["ground_truth"].isna().sum()))
print("\nGround-truth counts:")
print(merged["ground_truth"].value_counts(dropna=False))


## 9. Save outputs and audit files


In [ ]:
audit_cols = [
    prediction_id_col, "subject_key", "ground_truth",
    "clinical_resolution_status", "clinical_rows",
    "clinical_labels_observed", "cdr_values_observed"
]

for optional in ["prediction", "probability_positive", "correct"]:
    if optional in merged.columns:
        audit_cols.append(optional)

audit = merged[audit_cols].copy()

unmatched_predictions = merged.loc[
    merged["ground_truth"].isna(),
    [prediction_id_col, "subject_key"]
].drop_duplicates()

prediction_keys = set(merged["subject_key"].dropna())
unmatched_clinical = clinical_subjects.loc[
    ~clinical_subjects["subject_key"].isin(prediction_keys)
].copy()

merged.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
audit.to_csv(
    OUTPUT_DIR / "ground_truth_merge_audit.csv",
    index=False, encoding="utf-8-sig"
)
unmatched_predictions.to_csv(
    OUTPUT_DIR / "ground_truth_unmatched_predictions.csv",
    index=False, encoding="utf-8-sig"
)
unmatched_clinical.to_csv(
    OUTPUT_DIR / "ground_truth_unmatched_clinical.csv",
    index=False, encoding="utf-8-sig"
)

print("Saved:", OUTPUT_FILE)


## 10. Final validation


In [ ]:
total = len(merged)
matched = int(merged["ground_truth"].notna().sum())
coverage = matched / total if total else 0.0
classes = sorted(merged["ground_truth"].dropna().unique().tolist())

validation_passed = (
    total > 0
    and matched > 0
    and coverage >= 0.95
    and set(classes) == {"AD", "CN"}
)

report = {
    "step": "31B-auto-no-edit",
    "project_root": str(PROJECT_ROOT),
    "prediction_file": str(PREDICTION_FILE),
    "clinical_file": str(clinical_path),
    "output_file": str(OUTPUT_FILE),
    "prediction_records": int(total),
    "matched_ground_truth_records": int(matched),
    "coverage": float(coverage),
    "classes_found": classes,
    "validation_passed": bool(validation_passed),
    "label_rule": {
        "explicit_diagnosis_priority": True,
        "cdr_zero": "CN",
        "cdr_greater_than_zero": "AD"
    }
}

report_path = OUTPUT_DIR / "31B_ground_truth_integration_report.json"
report_path.write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print(f"Coverage: {matched}/{total} = {coverage:.2%}")
print("Classes:", classes)
print("Validation passed:", validation_passed)
print("Report:", report_path)

if validation_passed:
    print("\nSUCCESS — Ground-truth integration is ready.")
    print("Use this file in STEP 31A:")
    print(OUTPUT_FILE)
else:
    print("\nATTENTION — Review the unmatched/conflicting audit files before reporting metrics.")
